# 02 — IVF / IVF+PQ / IVF+SQ Benchmarks

We sweep the three Inverted-File index families that FAISS exposes:

* `IndexIVFFlat` — coarse quantiser + exact vectors per cell
* `IndexIVFPQ` — coarse quantiser + product-quantised residuals  (compressed)
* `IndexIVFScalarQuantizer` — coarse quantiser + per-component byte quantisation

The full base set (1 281 167 × 2048 D) is too large for a dense parameter grid in
reasonable wall-time, so we sweep on the **full base for IVFFlat** and use the **best
nlist** for the IVF+PQ / IVF+SQ comparison.

Metrics logged per configuration: build (s) · index size (MB) · QPS · R@1 · R@10 · R@100 ·
peak RSS during build (MB, background-sampled).

In [ ]:
import os, sys, time, gc, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import faiss
import psutil
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd()))
import utils

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

DATA = Path('data')
LAB_LIGHT = int(os.environ.get('LAB_LIGHT', '0'))
OUT_RUN = utils.run_mode()
RESULTS = utils.results_dir()
DOCS_IMG = utils.plots_dir()
print(f'OUT_RUN={OUT_RUN}  RESULTS={RESULTS}  DOCS_IMG={DOCS_IMG}')

# Load query + GT
with h5py.File(DATA / 'imagenet1m.h5', 'r') as h:
    queries = np.array(h['query'], dtype=np.float32)
    gt = np.array(h['groundtruth'], dtype=np.int32)
    DIM = int(h.attrs['dim']); N_BASE = int(h.attrs['n_base'])
    BASE_PATH = str(h.attrs['base_path'])

# Make BASE_PATH portable: prefer the file as it currently exists on disk
# (host path stored in h5 may not match container/CI paths).
_local = DATA / 'imagenet_base.fvecs'
if not Path(BASE_PATH).exists() and _local.exists():
    BASE_PATH = str(_local.resolve())
print('BASE_PATH =', BASE_PATH)

print('queries', queries.shape, 'gt', gt.shape, 'dim', DIM, 'base', N_BASE)
print(f"threads={faiss.omp_get_max_threads()}  RAM free={psutil.virtual_memory().available/1e9:.1f} GB")

In [ ]:
# ---------------------------------------------------------------------------
# Tunables — adjust to change the wall-time of this notebook.
# ---------------------------------------------------------------------------
# Subsample of the base used for parameter sweeps.  Set to N_BASE to use everything.
N_SWEEP = int(os.environ.get('LAB_N_SWEEP', 500_000))
LAB_LIGHT = int(os.environ.get('LAB_LIGHT', '0'))
# IVFFlat grid
NLIST_GRID  = [256, 1024, 4096, 16384]
NPROBE_GRID = [1, 4, 16, 64, 256, 1024]
# IVF+PQ grid — nlist × M (task requires varying coarse quantiser, not one nlist)
PQ_NLIST_GRID = [1024, 4096]
PQ_M_GRID = [32, 64, 128]   # divides 2048: 64,32,16 D per sub-vector
PQ_NBITS  = 8
# IVF+SQ grid
SQ_TYPES = [
    ('SQ8',  faiss.ScalarQuantizer.QT_8bit),
    ('SQ4',  faiss.ScalarQuantizer.QT_4bit),
]
SQ_NLIST_GRID = [256, 1024, 4096]

# Number of vectors used for IVF training (k-means).  ≥ 30 * nlist recommended.
TRAIN_N = max(200_000, 30 * max(NLIST_GRID))

QUERY_K = 100  # search depth; we report R@1, R@10, R@100

if LAB_LIGHT:
    TRAIN_N = min(TRAIN_N, 80_000)
    NLIST_GRID = [256, 1024]
    NPROBE_GRID = [1, 4, 16, 64, 256]
    PQ_NLIST_GRID = [256, 1024]
    PQ_M_GRID = [32, 64]
    SQ_TYPES = [('SQ8', faiss.ScalarQuantizer.QT_8bit)]
    SQ_NLIST_GRID = [256, 1024]

QPS_REPEAT = int(os.environ.get('LAB_QPS_REPEAT', '2' if LAB_LIGHT else '1'))
QPS_WARMUP = int(os.environ.get('LAB_QPS_WARMUP', '1' if LAB_LIGHT else '0'))
# Full runs: fewer queries per sweep cell keeps nbconvert under per-cell timeout.
_default_qn = queries.shape[0] if LAB_LIGHT else min(10000, queries.shape[0])
QUERY_N = int(os.environ.get('LAB_QUERY_N', str(_default_qn)))
queries_sweep = queries[:QUERY_N]
print(f"N_SWEEP={N_SWEEP:,}  TRAIN_N={TRAIN_N:,}  LAB_LIGHT={LAB_LIGHT}")
print(f"QPS_REPEAT={QPS_REPEAT}  QPS_WARMUP={QPS_WARMUP}  QUERY_N={QUERY_N}")
print(f"NLIST_GRID={NLIST_GRID}  NPROBE_GRID={NPROBE_GRID}")
print(f"PQ_NLIST_GRID={PQ_NLIST_GRID}  PQ_M_GRID={PQ_M_GRID}  SQ_TYPES={[t[0] for t in SQ_TYPES]}")
print(f"TRAIN_N={TRAIN_N:,}  faiss_threads={faiss.omp_get_max_threads()}")

## Helper — stream base vectors via memmap + recompute exact GT for the sweep subset

When `N_SWEEP < N_BASE` the supplied 100-NN ground truth is **invalid** for the subset
(it indexes into the full 1.28M base).  We recompute GT against the chosen subset using
`IndexFlatL2` — this is one-off, cached on disk and reused across notebooks.

In [ ]:
def ensure_gt(n: int, k: int = QUERY_K) -> np.ndarray:
    # Exact GT for first n base vectors; cached as data/gt_n{n}_k{k}.npy
    cache = DATA / f'gt_n{n}_k{k}.npy'
    if cache.exists():
        print(f'  ✓ cached GT  {cache}')
        return np.load(cache)
    print(f'Computing exact GT (Flat) on first {n:,} base vectors × {queries.shape[0]:,} queries × k={k}...')
    flat = faiss.IndexFlatL2(DIM)
    utils.stream_add(flat, BASE_PATH, n)
    _, I = flat.search(queries, k)
    np.save(cache, I)
    del flat; gc.collect()
    print(f'  cached → {cache}')
    return I

gt_local = ensure_gt(N_SWEEP)
# Small training slice only — the rest of the base is streamed straight into each index.
train_x = utils.load_train_subset(BASE_PATH, TRAIN_N)
utils.print_mem('after GT + train_x')
print('train_x', train_x.shape, 'gt_local', gt_local.shape)

## IVFFlat sweep — (nlist × nprobe)

Each `nlist` runs in its **own notebook cell** with CSV checkpointing so a 2h cell timeout
cannot discard the whole sweep.

In [ ]:
IVF_FLAT_PATH = RESULTS / 'ivf_flat.csv'
utils.init_results_csv(IVF_FLAT_PATH)
print('IVFFlat checkpoint:', IVF_FLAT_PATH)

#### IVFFlat — nlist=256

In [ ]:
if 256 not in NLIST_GRID:
    print('skip IVFFlat nlist=256 (not in NLIST_GRID)')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFFlat(quant, DIM, 256, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 256)
    with utils.timed('train+add nlist=256', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[nlist={256:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB · RSS {rss_mb:7.1f} MB · peak {rss_peak_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 256:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFFlat', nlist=256, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_FLAT_PATH)
    print(f'  → appended {len(rows)} rows to {IVF_FLAT_PATH}')

#### IVFFlat — nlist=1024

In [ ]:
if 1024 not in NLIST_GRID:
    print('skip IVFFlat nlist=1024 (not in NLIST_GRID)')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFFlat(quant, DIM, 1024, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 1024)
    with utils.timed('train+add nlist=1024', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[nlist={1024:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB · RSS {rss_mb:7.1f} MB · peak {rss_peak_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 1024:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFFlat', nlist=1024, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_FLAT_PATH)
    print(f'  → appended {len(rows)} rows to {IVF_FLAT_PATH}')

#### IVFFlat — nlist=4096

In [ ]:
if 4096 not in NLIST_GRID:
    print('skip IVFFlat nlist=4096 (not in NLIST_GRID)')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFFlat(quant, DIM, 4096, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 4096)
    with utils.timed('train+add nlist=4096', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[nlist={4096:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB · RSS {rss_mb:7.1f} MB · peak {rss_peak_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 4096:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFFlat', nlist=4096, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_FLAT_PATH)
    print(f'  → appended {len(rows)} rows to {IVF_FLAT_PATH}')

#### IVFFlat — nlist=16384

In [ ]:
if 16384 not in NLIST_GRID:
    print('skip IVFFlat nlist=16384 (not in NLIST_GRID)')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFFlat(quant, DIM, 16384, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 16384)
    with utils.timed('train+add nlist=16384', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[nlist={16384:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB · RSS {rss_mb:7.1f} MB · peak {rss_peak_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 16384:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFFlat', nlist=16384, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_FLAT_PATH)
    print(f'  → appended {len(rows)} rows to {IVF_FLAT_PATH}')

In [ ]:
df_ivf = pd.read_csv(IVF_FLAT_PATH)
if df_ivf.empty:
    raise RuntimeError('ivf_flat.csv has no rows — check IVFFlat sweep cells above')
display(df_ivf.tail(8))

### Plot 1 — Recall@100 vs nprobe (line per nlist)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for nl, sub in df_ivf.groupby('nlist'):
    sub = sub.sort_values('nprobe')
    ax.plot(sub.nprobe, sub.recall_100, marker='o', label=f'nlist={nl}')
ax.set_xscale('log'); ax.set_xlabel('nprobe'); ax.set_ylabel('Recall@100')
ax.set_title(f'IVFFlat — Recall@100 vs nprobe   (N_base={N_SWEEP:,})')
ax.legend(); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(DOCS_IMG / '02_ivf_recall_vs_nprobe.png', dpi=120); plt.show()

### Plot 2 — Heatmap nlist × nprobe → Recall@100

In [ ]:
pivot = df_ivf.pivot(index='nlist', columns='nprobe', values='recall_100')
# Invalid IVF configs: nprobe must be ≤ nlist — mask so heatmap shows gaps explicitly
mask_invalid = np.zeros_like(pivot, dtype=bool)
for i, nl in enumerate(pivot.index):
    for j, np_ in enumerate(pivot.columns):
        if int(np_) > int(nl):
            mask_invalid[i, j] = True
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap(pivot, mask=mask_invalid, annot=True, fmt='.3f', cmap='viridis', vmin=0, vmax=1,
            ax=ax, cbar_kws={'label': 'Recall@100'})
ax.set_title('IVFFlat — Recall@100 (masked where nprobe > nlist)')
plt.tight_layout(); plt.savefig(DOCS_IMG / '02_ivf_heatmap.png', dpi=120); plt.show()

### Plot 3 — QPS vs Recall@100 Pareto frontier

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_ivf.recall_100, df_ivf.qps, c=np.log2(df_ivf.nprobe.values+1),
           cmap='plasma', s=40, edgecolors='k')
mask = utils.pareto_frontier(df_ivf.recall_100.values, df_ivf.qps.values)
order = np.argsort(df_ivf.recall_100.values[mask])
ax.plot(df_ivf.recall_100.values[mask][order], df_ivf.qps.values[mask][order],
        'k--', lw=1, label='Pareto frontier')
# Annotate Pareto points only (avoids clutter)
dfm = df_ivf.iloc[np.where(mask)[0]].sort_values('recall_100')
for _, r in dfm.iterrows():
    ax.annotate(f"L{int(r.nlist)}/P{int(r.nprobe)}", (r.recall_100, r.qps),
                fontsize=7, alpha=0.85, xytext=(3, 3), textcoords='offset points')
ax.set_yscale('log'); ax.set_xlabel('Recall@100'); ax.set_ylabel('QPS (log)')
ax.set_title('IVFFlat — QPS vs Recall@100')
ax.legend()
plt.tight_layout(); plt.savefig(DOCS_IMG / '02_ivf_pareto.png', dpi=120); plt.show()

### Plot 4 — QPS vs nprobe (per nlist)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for nl, sub in df_ivf.groupby('nlist'):
    sub = sub.sort_values('nprobe')
    ax.plot(sub.nprobe, sub.qps, marker='o', label=f'nlist={nl}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('nprobe'); ax.set_ylabel('QPS (log)')
ax.set_title('IVFFlat — QPS vs nprobe')
ax.legend(); plt.tight_layout()
plt.savefig(DOCS_IMG / '02_ivf_qps_vs_nprobe.png', dpi=120); plt.show()

### Plot 5 — Build time vs nlist

In [ ]:
df_bt = df_ivf.drop_duplicates('nlist')[['nlist', 'build_s', 'size_mb']]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=df_bt, x='nlist', y='build_s', ax=ax[0], color='steelblue')
ax[0].set_yscale('log')
ax[0].set_title('IVFFlat build time (log)'); ax[0].set_ylabel('seconds')
sns.barplot(data=df_bt, x='nlist', y='size_mb', ax=ax[1], color='darkorange')
ax[1].set_title('IVFFlat index size'); ax[1].set_ylabel('MB')
plt.tight_layout(); plt.savefig(DOCS_IMG / '02_ivf_build_size.png', dpi=120); plt.show()
display(df_bt.reset_index(drop=True))

## IVF+PQ sweep — nlist × PQ M

Full grid over `PQ_NLIST_GRID × PQ_M_GRID` (not a single best nlist from IVFFlat).

In [ ]:
IVF_PQ_PATH = RESULTS / 'ivf_pq.csv'
utils.init_results_csv(IVF_PQ_PATH)
print('IVFPQ checkpoint:', IVF_PQ_PATH)

# SQ sweep is now multi-nlist (SQ_NLIST_GRID); keep best_nlist as the *reference*
# IVFFlat point used in cross-family Pareto overlays only.
best_nlist = int((df_ivf
              .groupby('nlist')['recall_100'].max()
              .idxmax()))
print(f'reference IVFFlat nlist for plots: {best_nlist}  ·  SQ_NLIST_GRID={SQ_NLIST_GRID}')

#### IVF+PQ — nlist=1024 M=32

In [ ]:
if 1024 not in PQ_NLIST_GRID or 32 not in PQ_M_GRID:
    print('skip IVFPQ nlist=1024 M=32')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFPQ(quant, DIM, 1024, int(32), int(PQ_NBITS))
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 1024)
    with utils.timed('train+add PQ nlist=1024 M=32', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[PQ nlist={1024:5} M={32:4}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 1024:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFPQ', nlist=1024, nprobe=nprobe, M=32, nbits=PQ_NBITS,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_PQ_PATH)

#### IVF+PQ — nlist=1024 M=64

In [ ]:
if 1024 not in PQ_NLIST_GRID or 64 not in PQ_M_GRID:
    print('skip IVFPQ nlist=1024 M=64')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFPQ(quant, DIM, 1024, int(64), int(PQ_NBITS))
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 1024)
    with utils.timed('train+add PQ nlist=1024 M=64', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[PQ nlist={1024:5} M={64:4}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 1024:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFPQ', nlist=1024, nprobe=nprobe, M=64, nbits=PQ_NBITS,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_PQ_PATH)

#### IVF+PQ — nlist=1024 M=128

In [ ]:
if 1024 not in PQ_NLIST_GRID or 128 not in PQ_M_GRID:
    print('skip IVFPQ nlist=1024 M=128')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFPQ(quant, DIM, 1024, int(128), int(PQ_NBITS))
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 1024)
    with utils.timed('train+add PQ nlist=1024 M=128', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[PQ nlist={1024:5} M={128:4}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 1024:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFPQ', nlist=1024, nprobe=nprobe, M=128, nbits=PQ_NBITS,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_PQ_PATH)

#### IVF+PQ — nlist=4096 M=32

In [ ]:
if 4096 not in PQ_NLIST_GRID or 32 not in PQ_M_GRID:
    print('skip IVFPQ nlist=4096 M=32')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFPQ(quant, DIM, 4096, int(32), int(PQ_NBITS))
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 4096)
    with utils.timed('train+add PQ nlist=4096 M=32', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[PQ nlist={4096:5} M={32:4}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 4096:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFPQ', nlist=4096, nprobe=nprobe, M=32, nbits=PQ_NBITS,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_PQ_PATH)

#### IVF+PQ — nlist=4096 M=64

In [ ]:
if 4096 not in PQ_NLIST_GRID or 64 not in PQ_M_GRID:
    print('skip IVFPQ nlist=4096 M=64')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFPQ(quant, DIM, 4096, int(64), int(PQ_NBITS))
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 4096)
    with utils.timed('train+add PQ nlist=4096 M=64', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[PQ nlist={4096:5} M={64:4}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 4096:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFPQ', nlist=4096, nprobe=nprobe, M=64, nbits=PQ_NBITS,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_PQ_PATH)

#### IVF+PQ — nlist=4096 M=128

In [ ]:
if 4096 not in PQ_NLIST_GRID or 128 not in PQ_M_GRID:
    print('skip IVFPQ nlist=4096 M=128')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFPQ(quant, DIM, 4096, int(128), int(PQ_NBITS))
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 4096)
    with utils.timed('train+add PQ nlist=4096 M=128', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[PQ nlist={4096:5} M={128:4}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 4096:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFPQ', nlist=4096, nprobe=nprobe, M=128, nbits=PQ_NBITS,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_PQ_PATH)

In [ ]:
df_pq = pd.read_csv(IVF_PQ_PATH)
display(df_pq.tail(6))

### Plot 6 — IVF+PQ Recall@100 vs nprobe (per M)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for (nl, M), sub in df_pq.groupby(['nlist', 'M']):
    sub = sub.sort_values('nprobe')
    ax.plot(sub.nprobe, sub.recall_100, marker='o', label=f'nlist={int(nl)} M={int(M)}')
ax.set_xscale('log'); ax.set_xlabel('nprobe'); ax.set_ylabel('Recall@100')
ax.set_title('IVF+PQ — Recall@100 vs nprobe (per nlist × M)')
ax.legend(); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(DOCS_IMG / '02_ivfpq_recall.png', dpi=120); plt.show()

### Plot 7 — IVF+PQ Pareto QPS vs Recall@100 (per M)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
markers = ['o', 's', '^', 'D', 'v', 'P']
for i, (nl, M) in enumerate(sorted({(int(r.nlist), int(r.M)) for _, r in df_pq.iterrows()})):
    sub = df_pq[(df_pq.nlist == nl) & (df_pq.M == M)]
    mk = markers[i % len(markers)]
    ax.scatter(sub.recall_100, sub.qps, marker=mk, s=45, edgecolors='k',
               label=f'nlist={nl} M={M}')
    ms = utils.pareto_frontier(sub.recall_100.values, sub.qps.values)
    if ms.any():
        o = np.argsort(sub.recall_100.values[ms])
        ax.plot(sub.recall_100.values[ms][o], sub.qps.values[ms][o], ls='--', lw=0.8, alpha=0.7)
ax.scatter(df_ivf[df_ivf.nlist==best_nlist].recall_100, df_ivf[df_ivf.nlist==best_nlist].qps,
           marker='X', color='k', s=90, label=f'IVFFlat nlist={best_nlist}', zorder=5)
ax.set_yscale('log'); ax.set_xlabel('Recall@100'); ax.set_ylabel('QPS (log)')
ax.set_title('IVF+PQ — QPS vs Recall@100 (each nlist × M)')
ax.legend(fontsize=7, ncol=2, loc='lower right')
plt.tight_layout(); plt.savefig(DOCS_IMG / '02_ivfpq_pareto.png', dpi=120); plt.show()

### Plot 8 — Index size vs PQ M (compression vs flat)

In [ ]:
size_rows = [{'family': 'IVFFlat', 'nlist': best_nlist, 'M': None,
              'size_mb': float(df_ivf[df_ivf.nlist==best_nlist].size_mb.iloc[0])}]
for nl in sorted(df_pq.nlist.unique()):
    for M in sorted(df_pq.M.unique()):
        sub = df_pq[(df_pq.nlist == nl) & (df_pq.M == M)]
        size_rows.append({'family': 'IVFPQ', 'nlist': int(nl), 'M': int(M),
                         'size_mb': float(sub.size_mb.iloc[0])})
size_df = pd.DataFrame(size_rows)
size_df['label'] = size_df.apply(
    lambda r: f"Flat L{int(r.nlist)}" if r.family == 'IVFFlat' else f"PQ L{int(r.nlist)} M{int(r.M)}",
    axis=1)
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=size_df, x='label', y='size_mb', hue='family', ax=ax, dodge=False)
ax.set_yscale('log'); ax.set_ylabel('MB (log)')
ax.set_title('Index footprint — IVFFlat vs IVF+PQ (per nlist × M)')
for i, (_, r) in enumerate(size_df.iterrows()):
    ax.text(i, r.size_mb, f'{r.size_mb:.0f}', ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=25, ha='right')
plt.tight_layout(); plt.savefig(DOCS_IMG / '02_ivfpq_size.png', dpi=120); plt.show()
display(size_df)

## IVF+ScalarQuantizer

One SQ variant per notebook cell (checkpointed CSV).

In [ ]:
IVF_SQ_PATH = RESULTS / 'ivf_sq.csv'
utils.init_results_csv(IVF_SQ_PATH)
print('IVFSQ checkpoint:', IVF_SQ_PATH)

#### IVF+SQ — nlist=256 · SQ8

In [ ]:
if 'SQ8' not in [t[0] for t in SQ_TYPES] or 256 not in SQ_NLIST_GRID:
    print('skip IVFSQ SQ8 nlist=256')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFScalarQuantizer(quant, DIM, 256, faiss.ScalarQuantizer.QT_8bit, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 256)
    with utils.timed('train+add SQ SQ8 nlist=256', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[SQ SQ8 nlist={256:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 256:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFSQ', sq='SQ8', nlist=256, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_SQ_PATH)

#### IVF+SQ — nlist=256 · SQ4

In [ ]:
if 'SQ4' not in [t[0] for t in SQ_TYPES] or 256 not in SQ_NLIST_GRID:
    print('skip IVFSQ SQ4 nlist=256')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFScalarQuantizer(quant, DIM, 256, faiss.ScalarQuantizer.QT_4bit, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 256)
    with utils.timed('train+add SQ SQ4 nlist=256', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[SQ SQ4 nlist={256:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 256:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFSQ', sq='SQ4', nlist=256, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_SQ_PATH)

#### IVF+SQ — nlist=1024 · SQ8

In [ ]:
if 'SQ8' not in [t[0] for t in SQ_TYPES] or 1024 not in SQ_NLIST_GRID:
    print('skip IVFSQ SQ8 nlist=1024')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFScalarQuantizer(quant, DIM, 1024, faiss.ScalarQuantizer.QT_8bit, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 1024)
    with utils.timed('train+add SQ SQ8 nlist=1024', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[SQ SQ8 nlist={1024:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 1024:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFSQ', sq='SQ8', nlist=1024, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_SQ_PATH)

#### IVF+SQ — nlist=1024 · SQ4

In [ ]:
if 'SQ4' not in [t[0] for t in SQ_TYPES] or 1024 not in SQ_NLIST_GRID:
    print('skip IVFSQ SQ4 nlist=1024')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFScalarQuantizer(quant, DIM, 1024, faiss.ScalarQuantizer.QT_4bit, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 1024)
    with utils.timed('train+add SQ SQ4 nlist=1024', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[SQ SQ4 nlist={1024:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 1024:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFSQ', sq='SQ4', nlist=1024, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_SQ_PATH)

#### IVF+SQ — nlist=4096 · SQ8

In [ ]:
if 'SQ8' not in [t[0] for t in SQ_TYPES] or 4096 not in SQ_NLIST_GRID:
    print('skip IVFSQ SQ8 nlist=4096')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFScalarQuantizer(quant, DIM, 4096, faiss.ScalarQuantizer.QT_8bit, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 4096)
    with utils.timed('train+add SQ SQ8 nlist=4096', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[SQ SQ8 nlist={4096:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 4096:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFSQ', sq='SQ8', nlist=4096, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_SQ_PATH)

#### IVF+SQ — nlist=4096 · SQ4

In [ ]:
if 'SQ4' not in [t[0] for t in SQ_TYPES] or 4096 not in SQ_NLIST_GRID:
    print('skip IVFSQ SQ4 nlist=4096')
else:
    rows = []
    quant = faiss.IndexFlatL2(DIM)
    idx = faiss.IndexIVFScalarQuantizer(quant, DIM, 4096, faiss.ScalarQuantizer.QT_4bit, faiss.METRIC_L2)
    idx.cp.min_points_per_centroid = 5
    idx.cp.max_points_per_centroid = max(256, len(train_x) // 4096)
    with utils.timed('train+add SQ SQ4 nlist=4096', sample_rss_peak=True) as tb:
        idx.train(train_x)
        utils.stream_add(idx, BASE_PATH, N_SWEEP)
    size_mb = utils.index_size_mb(idx)
    rss_mb = tb.rss_after_mb
    rss_peak_mb = tb.rss_peak_mb
    rss_delta_mb = tb.rss_delta_mb
    print(f'[SQ SQ4 nlist={4096:5}]  build {tb.elapsed:6.1f}s · size {size_mb:7.1f} MB')
    for nprobe in NPROBE_GRID:
        if nprobe > 4096:
            continue
        idx.nprobe = nprobe
        qps, lat_ms, lat_p99_ms, I = utils.measure_qps(
            lambda q,k: idx.search(q,k), queries_sweep, QUERY_K,
            repeat=QPS_REPEAT, warmup=QPS_WARMUP,
        )
        recalls = utils.compute_recalls(I, gt_local[:QUERY_N], (1, 10, 100))
        rows.append(dict(algo='IVFSQ', sq='SQ4', nlist=4096, nprobe=nprobe,
                        build_s=tb.elapsed, size_mb=size_mb, rss_mb=rss_mb, rss_peak_mb=rss_peak_mb,
                        rss_delta_mb=rss_delta_mb, **utils.bench_meta(),
                        qps=qps, latency_ms=lat_ms, latency_p99_ms=lat_p99_ms,
                        recall_1=recalls[1], recall_10=recalls[10], recall_100=recalls[100],
                        n_base=N_SWEEP))
        print(f'    nprobe={nprobe:5}  qps={qps:8.1f}  R@100={recalls[100]:.3f}')
    del idx, quant; gc.collect()
    utils.append_results(rows, IVF_SQ_PATH)

In [ ]:
df_sq = pd.read_csv(IVF_SQ_PATH)
display(df_sq)

### Plot 9 — SQ vs PQ vs Flat at the same nlist (Pareto)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(df_ivf[df_ivf.nlist==best_nlist].recall_100, df_ivf[df_ivf.nlist==best_nlist].qps,
           label=f'IVFFlat nlist={best_nlist}', color='black', s=55, marker='X')
for M, sub in df_pq.groupby('M'):
    ax.scatter(sub.recall_100, sub.qps, label=f'IVF+PQ M={M}', s=40, alpha=0.85)
for name, sub in df_sq.groupby('sq'):
    ax.scatter(sub.recall_100, sub.qps, label=f'IVF+{name}', s=70, marker='s', alpha=0.9)
ax.set_yscale('log'); ax.set_xlabel('Recall@100'); ax.set_ylabel('QPS (log)')
ax.set_title('IVF variants — Pareto QPS vs Recall@100')
ax.legend()
plt.tight_layout(); plt.savefig(DOCS_IMG / '02_ivf_all_pareto.png', dpi=120); plt.show()

# Release training slice (~1.6 GB) before notebooks save and plots render.
try:
    del train_x
except NameError:
    pass
gc.collect()
utils.print_mem('after IVF sweeps')

# Persist combined for notebook 05
df_all = pd.concat([df_ivf, df_pq, df_sq], ignore_index=True)
df_all.to_csv(RESULTS / 'ivf_all.csv', index=False)
print(f'rows={len(df_all)}  → {RESULTS / "ivf_all.csv"}')

## Summary

Notable observations to keep in mind when reading `05_comparison.ipynb`:

* The IVFFlat upper bound on Recall@100 is set by `nprobe / nlist`. At the largest
  `nlist`/`nprobe` ratio we should saturate to ≥ 0.99.
* IVF+PQ trades index size for recall — index footprint drops by ~50–100× while
  recall stays within ~10–20 % of Flat **only at high nprobe**.
* IVF+ScalarQuantizer is a middle ground — 4× compression with minimal recall loss.